# 50 — Generate `usdm_v4.shapes.ttl` + `usdm_v4.shapes-ct.ttl`

SHACL shapes for USDM v4 instance validation, generated mechanically from
the same pinned sources as the other deliverables. Two layers, two files:

- **`usdm_v4.shapes.ttl` — structural layer.** One closed `sh:NodeShape`
  per concrete class (flattened: constraints for all attributes including
  inherited, on their declaring-class property IRIs). Validates a bare
  instance graph as produced by `usdm_v4.context.jsonld` — no ontology
  merge, no inference.
- **`usdm_v4.shapes-ct.ttl` — terminology layer.** For the 25 DDF-native
  value sets published in sheet 2 of `USDM_CT.xlsx`: `sh:in` over the
  permitted Concept C-codes on the `Code` values of the bound property,
  targeted via `sh:targetObjectsOf`. Severity from the published
  extensibility flag: non-extensible → `sh:Violation`, extensible →
  `sh:Warning`.

Source boundary: everything derivable from the two pinned source files is
used fully; external terminology packages (SDTM/Protocol Terminology) and
free-text dictionary references stay out. Design rationale in
`docs/shacl-design.md`; IRI aspects are decision D6 in
`docs/iri-and-governance.md`.


## 1. Load source YAML and USDM_CT value sets


In [1]:
import yaml
import openpyxl
from pathlib import Path

YAML_PATH = "../downloads/dataStructure.yml"
CT_PATH = "../downloads/USDM_CT.xlsx"
for p in (YAML_PATH, CT_PATH):
    if not Path(p).exists():
        raise FileNotFoundError(
            f"{p} missing — run notebooks/10_fetch_yaml.ipynb first."
        )

with open(YAML_PATH) as f:
    MODEL = yaml.safe_load(f)

wb = openpyxl.load_workbook(CT_PATH, read_only=True, data_only=True)
ws = wb["DDF valid value sets"]
rows = list(ws.iter_rows(values_only=True))
header = list(rows[5])
EXPECTED_HEADER = [
    "Project", "Entity", "Attribute", "Codelist C-code",
    "Codelist Extensibility (Yes/No)", "Concept C-code",
    "Preferred Term (CDISC Submission Value)",
]
if header[:7] != EXPECTED_HEADER:
    raise ValueError(f"sheet 2 header drift: {header[:7]}")

VALUE_SETS = {}
for r in rows[6:]:
    if r[0] != "DDF":
        continue
    key = (r[1], r[2], r[3], r[4])
    VALUE_SETS.setdefault(key, []).append(r[5])

print(f"{len(MODEL)} top-level YAML entries")
print(f"{len(VALUE_SETS)} value sets, {sum(map(len, VALUE_SETS.values()))} permitted values")


86 top-level YAML entries
25 value sets, 125 permitted values


## 2. Declaring-class resolution

Same rule as `40_generate_context.ipynb`: follow the `Inherited From`
chain to the declaring class, whose property IRI the instance data
carries. A fail-fast guard asserts that redeclared rows never diverge
from their declaring entry in `Cardinality`, `Type`, or
`Relationship Type` — if a future DDF-RA release introduces covariant
redeclaration, this notebook stops rather than emitting shapes that
silently use the wrong constraint.


In [2]:
ABSTRACT = {cn for cn, e in MODEL.items() if e.get("Modifier") == "Abstract"}
CONCRETE = sorted(cn for cn in MODEL if cn not in ABSTRACT)


def resolve_ref(ref):
    return ref.lstrip("#/")


def super_classes(class_name):
    entry = MODEL[class_name]
    return [resolve_ref(r["$ref"]) for r in (entry.get("Super Classes") or [])]


def all_keys(class_name):
    keys = set(MODEL[class_name].get("Attributes") or {})
    for sc in super_classes(class_name):
        keys |= all_keys(sc)
    return keys


def declaring_class(class_name, key):
    attrs = MODEL[class_name].get("Attributes") or {}
    if key in attrs:
        inherited_from = attrs[key].get("Inherited From")
        if not inherited_from:
            return class_name
        if len(inherited_from) != 1:
            raise ValueError(
                f"{class_name}.{key}: multi-parent Inherited From {inherited_from}"
            )
        return declaring_class(resolve_ref(inherited_from[0]["$ref"]), key)
    hits = {
        declaring_class(sc, key)
        for sc in super_classes(class_name)
        if key in all_keys(sc)
    }
    if len(hits) != 1:
        raise KeyError(f"{class_name}.{key}: declaring class not unique: {sorted(hits)}")
    return hits.pop()


def concrete_descendants(cls):
    out = set()
    for cn in MODEL:
        if cls in super_classes(cn):
            if cn in ABSTRACT:
                out |= concrete_descendants(cn)
            else:
                out.add(cn)
                out |= concrete_descendants(cn)
    return out


for cn in MODEL:
    for key, attr in (MODEL[cn].get("Attributes") or {}).items():
        if "Inherited From" not in attr:
            continue
        decl = declaring_class(cn, key)
        decl_attr = MODEL[decl]["Attributes"][key]
        for field in ("Cardinality", "Type", "Relationship Type"):
            if attr.get(field) != decl_attr.get(field):
                raise ValueError(
                    f"{cn}.{key} diverges from {decl}.{key} on {field}: "
                    f"{attr.get(field)!r} vs {decl_attr.get(field)!r}"
                )

print(f"{len(CONCRETE)} concrete + {len(ABSTRACT)} abstract classes; no redeclaration divergence")


80 concrete + 6 abstract classes; no redeclaration divergence


## 3. Structural shapes (flatten + closed)

Per concrete class: `sh:targetClass`, `sh:closed true`,
`sh:ignoredProperties (rdf:type)`, and one `sh:property` per flattened
attribute. `{Class}-id` and `{Class}-instanceType` are excluded — the
instance context absorbs them into `@id`/`@type`, so closed shapes permit
exactly the predicates the context can produce.

| Source                          | Constraint                                     |
|---------------------------------|------------------------------------------------|
| Cardinality `1`                 | `sh:minCount 1 ; sh:maxCount 1`                |
| Cardinality `0..1`              | `sh:maxCount 1`                                |
| Cardinality `1..*`              | `sh:minCount 1`                                |
| Cardinality `0..*`              | (none)                                         |
| Cardinality `0..2`              | `sh:maxCount 2`                                |
| primitive Type                  | `sh:datatype xsd:*`                            |
| concrete class Type             | `sh:class`                                     |
| abstract class Type             | `sh:or` over concrete descendants              |
| multi-`$ref` Type (union)       | `sh:or` over members (abstract members expanded) |
| `Relationship Type: Ref`        | additionally `sh:nodeKind sh:IRI`              |

Any cardinality form outside the five observed raises.


In [3]:
PRIM_TO_XSD = {
    "string": "xsd:string",
    "boolean": "xsd:boolean",
    "date": "xsd:date",
    "float": "xsd:float",
    "integer": "xsd:integer",
}
CARDINALITY = {
    "1": ("sh:minCount 1", "sh:maxCount 1"),
    "0..1": (None, "sh:maxCount 1"),
    "1..*": ("sh:minCount 1", None),
    "0..*": (None, None),
    "0..2": (None, "sh:maxCount 2"),
}


def range_expression(refs):
    targets = []
    for r in refs:
        if r in ABSTRACT:
            targets.extend(sorted(concrete_descendants(r)))
        else:
            targets.append(r)
    if len(targets) == 1:
        return f"sh:class usdm:{targets[0]}", 0
    alternatives = " ".join(f"[ sh:class usdm:{t} ]" for t in targets)
    return f"sh:or ( {alternatives} )", 1


core_lines = []
n_property_shapes = 0
n_or = 0
for cn in CONCRETE:
    core_lines.append(f"usdm:{cn}-shape")
    core_lines.append("    a sh:NodeShape ;")
    core_lines.append(f"    sh:targetClass usdm:{cn} ;")
    core_lines.append("    sh:closed true ;")
    core_lines.append("    sh:ignoredProperties ( rdf:type ) ;")
    for key in sorted(all_keys(cn)):
        if key in ("id", "instanceType"):
            continue
        decl = declaring_class(cn, key)
        attr = MODEL[decl]["Attributes"][key]
        refs = [resolve_ref(t["$ref"]) for t in attr["Type"]]
        parts = [f"sh:path usdm:{decl}-{key}"]
        card = attr.get("Cardinality")
        if card not in CARDINALITY:
            raise ValueError(f"unmapped cardinality {card!r} on {decl}.{key}")
        min_c, max_c = CARDINALITY[card]
        if min_c:
            parts.append(min_c)
        if max_c:
            parts.append(max_c)
        if all(r in PRIM_TO_XSD for r in refs):
            if len(refs) != 1:
                raise ValueError(f"multi-primitive Type on {decl}.{key}: {refs}")
            parts.append(f"sh:datatype {PRIM_TO_XSD[refs[0]]}")
        else:
            if any(r in PRIM_TO_XSD for r in refs):
                raise ValueError(f"mixed primitive/class Type on {decl}.{key}: {refs}")
            if attr.get("Relationship Type") == "Ref":
                parts.append("sh:nodeKind sh:IRI")
            expression, is_or = range_expression(refs)
            n_or += is_or
            parts.append(expression)
        core_lines.append("    sh:property [ " + " ; ".join(parts) + " ] ;")
        n_property_shapes += 1
    core_lines[-1] = core_lines[-1].rstrip(" ;") + " ."
    core_lines.append("")

CORE_BASELINE = {"node_shapes": 80, "property_shapes": 619, "sh_or": 20}
observed = {"node_shapes": len(CONCRETE), "property_shapes": n_property_shapes, "sh_or": n_or}
for k, expected in CORE_BASELINE.items():
    assert observed[k] == expected, f"{k}: expected {expected}, got {observed[k]}"
print(observed)


{'node_shapes': 80, 'property_shapes': 619, 'sh_or': 20}


## 4. Terminology shapes from sheet 2

One shape per published value set, targeting the `Code` values of the
bound property via `sh:targetObjectsOf usdm:{DeclaringClass}-{attribute}`.
The codelist C-code is cross-checked against the sheet-1 binding for the
same attribute where one exists — disagreement raises. `Code`-typed
attributes constrain `usdm:Code-code`; `AliasCode`-typed attributes would
constrain the sequence path via `usdm:AliasCode-standardCode` (none occur
in DDF-RA v4.0.0); any other type raises.


In [4]:
import re

ws1 = wb["DDF Entities&Attributes"]
rows1 = list(ws1.iter_rows(values_only=True))
header1 = list(rows1[0])
i_entity = header1.index("Entity Name")
i_attr = header1.index("Logical Data Model Name")
i_url = header1.index("Codelist URL")
i_inh = header1.index("Inherited From")
RE_URL_CCODE = re.compile(r"/subset/ncit/(C\d+)\s*$")
SHEET1_CCODE = {}
for r in rows1[1:]:
    if r[i_inh] or not r[i_url]:
        continue
    m = RE_URL_CCODE.search(str(r[i_url]))
    if m:
        SHEET1_CCODE[(r[i_entity], r[i_attr])] = m.group(1)
if not SHEET1_CCODE:
    raise ValueError("sheet 1 codelist cross-check index is empty — regex or column drift")

ct_lines = []
n_violation = 0
n_warning = 0
for (entity, attr_name, cl_ccode, extensible), values in sorted(VALUE_SETS.items()):
    decl = declaring_class(entity, attr_name)
    attr = MODEL[decl]["Attributes"][attr_name]
    refs = [resolve_ref(t["$ref"]) for t in attr["Type"]]
    if len(refs) != 1:
        raise ValueError(f"{entity}.{attr_name}: unexpected multi-ref Type {refs}")
    if refs[0] == "Code":
        path = "usdm:Code-code"
    elif refs[0] == "AliasCode":
        path = "( usdm:AliasCode-standardCode usdm:Code-code )"
    else:
        raise ValueError(f"{entity}.{attr_name}: unexpected type {refs[0]}")
    sheet1 = SHEET1_CCODE.get((entity, attr_name))
    if sheet1 is not None and sheet1 != cl_ccode:
        raise ValueError(
            f"{entity}.{attr_name}: sheet 2 codelist {cl_ccode} != sheet 1 {sheet1}"
        )
    if extensible == "Yes":
        severity = "sh:Warning"
        n_warning += 1
    elif extensible == "No":
        severity = "sh:Violation"
        n_violation += 1
    else:
        raise ValueError(f"{entity}.{attr_name}: extensibility {extensible!r}")
    in_list = " ".join(f'"{v}"' for v in values)
    ct_lines.append(f"# codelist {cl_ccode}, extensible: {extensible}")
    ct_lines.append(f"usdm:{decl}-{attr_name}-ct-shape")
    ct_lines.append("    a sh:NodeShape ;")
    ct_lines.append(f"    sh:targetObjectsOf usdm:{decl}-{attr_name} ;")
    ct_lines.append(
        f"    sh:property [ sh:path {path} ; sh:in ( {in_list} ) ; "
        f"sh:severity {severity} ] ."
    )
    ct_lines.append("")

CT_BASELINE = {"ct_shapes": 25, "values": 125, "violation_severity": 9, "warning_severity": 16}
observed_ct = {
    "ct_shapes": len(VALUE_SETS),
    "values": sum(map(len, VALUE_SETS.values())),
    "violation_severity": n_violation,
    "warning_severity": n_warning,
}
for k, expected in CT_BASELINE.items():
    assert observed_ct[k] == expected, f"{k}: expected {expected}, got {observed_ct[k]}"
print(observed_ct)


{'ct_shapes': 25, 'values': 125, 'violation_severity': 9, 'warning_severity': 16}


## 5. Write deliverables


In [5]:
RELEASE_VERSION = "v0.6.0"

PREFIXES = """@prefix sh: <http://www.w3.org/ns/shacl#> .
@prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .
@prefix owl: <http://www.w3.org/2002/07/owl#> .
@prefix dcterms: <http://purl.org/dc/terms/> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .
@prefix usdm: <https://w3id.org/cdisc/usdm/v4/> .
"""


def graph_header(iri_suffix, title):
    return (
        f"<https://w3id.org/cdisc/usdm/v4/{iri_suffix}>\n"
        "    a owl:Ontology ;\n"
        f'    dcterms:title "{title}" ;\n'
        '    dcterms:source <https://w3id.org/cdisc/usdm/v4/> ;\n'
        "    dcterms:license <https://creativecommons.org/licenses/by/4.0/> ;\n"
        f'    owl:versionInfo "{RELEASE_VERSION}" .\n'
    )


with open("../usdm_v4.shapes.ttl", "w") as f:
    f.write(PREFIXES + "\n")
    f.write(graph_header("shapes", "CDISC USDM v4 structural SHACL shapes") + "\n")
    f.write("\n".join(core_lines))

with open("../usdm_v4.shapes-ct.ttl", "w") as f:
    f.write(PREFIXES + "\n")
    f.write(graph_header("shapes-ct", "CDISC USDM v4 terminology SHACL shapes (DDF value sets)") + "\n")
    f.write("\n".join(ct_lines))

print("wrote ../usdm_v4.shapes.ttl and ../usdm_v4.shapes-ct.ttl")


wrote ../usdm_v4.shapes.ttl and ../usdm_v4.shapes-ct.ttl


## 6. Self-check against `usdm_v4.ttl` and SHACL meta-validation

Every `sh:path` IRI (including sequence-path members) and every
`sh:class` target must be declared in the ontology; both shapes graphs
must themselves conform to SHACL-SHACL (pyshacl `meta_shacl`).


In [6]:
import rdflib
from rdflib.namespace import OWL, RDF, SH
from pyshacl import validate

TTL_PATH = "../usdm_v4.ttl"
if not Path(TTL_PATH).exists():
    raise FileNotFoundError(
        f"{TTL_PATH} missing — run notebooks/20_generate_turtle.ipynb first."
    )
ont = rdflib.Graph()
ont.parse(TTL_PATH, format="turtle")
declared_props = (
    set(ont.subjects(RDF.type, OWL.DatatypeProperty))
    | set(ont.subjects(RDF.type, OWL.ObjectProperty))
)
declared_classes = {
    c for c in ont.subjects(RDF.type, OWL.Class) if isinstance(c, rdflib.URIRef)
}

for name in ("../usdm_v4.shapes.ttl", "../usdm_v4.shapes-ct.ttl"):
    g = rdflib.Graph()
    g.parse(name, format="turtle")
    paths = set()
    for p in g.objects(None, SH.path):
        if isinstance(p, rdflib.URIRef):
            paths.add(p)
        else:
            paths.update(x for x in g.items(p) if isinstance(x, rdflib.URIRef))
    bad_paths = paths - declared_props
    assert not bad_paths, f"{name}: undeclared sh:path IRIs: {sorted(bad_paths)[:5]}"
    bad_classes = set(g.objects(None, SH["class"])) - declared_classes
    assert not bad_classes, f"{name}: undeclared sh:class targets: {sorted(bad_classes)[:5]}"
    conforms, _, _ = validate(
        data_graph=rdflib.Graph(), shacl_graph=g, meta_shacl=True, inference="none"
    )
    assert conforms, f"{name}: shapes graph fails SHACL-SHACL meta-validation"
    n_targets = len(set(g.subjects(RDF.type, SH.NodeShape)))
    print(f"{name}: {len(g)} triples, {n_targets} node shapes, paths/classes declared, meta-SHACL ok")


../usdm_v4.shapes.ttl: 3135 triples, 80 node shapes, paths/classes declared, meta-SHACL ok


../usdm_v4.shapes-ct.ttl: 407 triples, 25 node shapes, paths/classes declared, meta-SHACL ok


## Provenance

Generated by `notebooks/50_generate_shapes.ipynb` in
[kerfors/usdm-rdf](https://github.com/kerfors/usdm-rdf) from
`downloads/dataStructure.yml` and `downloads/USDM_CT.xlsx` (DDF-RA tag
pinned in `notebooks/10_fetch_yaml.ipynb`).
